# 10 - Object-Oriented Programming (OOP)


## 1. What is OOP & Classes & Objects

Nesne Yönelimli Programlama (OOP), veriyi (nitelikler) ve bu veri üzerinde işlem yapan davranışları (metotlar) tek bir yapıda birleştiren paradigmadır. Sınıf (`class`) bir taslak iken, nesne (`object`) bu taslaktan üretilen somut örnektir.

**Syntax**
```python
class ClassName:
    def method_name(self) -> ReturnType:
        ...

obj: ClassName = ClassName()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Benzer varlıkların kendi durumunu (state) ve davranışlarını bağımsız şekilde yönetmesi gereken yapılarda kod tekrarını önlemek için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Durum tutmayan basit veri dönüşümleri için gereksiz sınıf hiyerarşileri kurmak yerine modül düzeyinde fonksiyonlar tercih edilmelidir.


In [ ]:
class AudioTrack:
    def __init__(self, title: str, bpm: int) -> None:
        self.title = title
        self.bpm = bpm
        self.is_playing = False

    def play(self) -> None:
        self.is_playing = True
        print(f'Playing "{self.title}" at {self.bpm} BPM')

    def stop(self) -> None:
        self.is_playing = False
        print(f'Stopped "{self.title}"')


track_a: AudioTrack = AudioTrack('Solea Por Bulerias', 190)
track_b: AudioTrack = AudioTrack('Midnight Blues', 92)

track_a.play()
track_b.play()
track_a.stop()


## 2. What is `self`?

`self`, bir örnek metodunun (instance method) o anda hangi somut nesne üzerinde çağrıldığını gösteren referanstır.

**Syntax**
```python
class ClassName:
    def method(self, param: type) -> None:
        self.attribute = param
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Tüm örnek metotlarının ilk parametresi olarak yazılır; Python metot çağrısında nesneyi bu parametreye otomatik iletir.
* **Dikkat Edilmesi Gerekenler:** PEP 8 standardı gereği her zaman `self` adı kullanılmalıdır. Metot içinde `self.` öneki olmadan tanımlanan değişkenler nesneye bağlanmaz, yerel değişken olarak kalır.


In [ ]:
class ChannelStrip:
    def __init__(self, name: str, gain_db: float) -> None:
        self.name = name
        self.gain_db = gain_db

    def boost(self, amount: float) -> None:
        self.gain_db += amount
        print(f'{self.name}: {self.gain_db} dB')


vocal: ChannelStrip = ChannelStrip('Lead Vocal', 0.0)
guitar: ChannelStrip = ChannelStrip('Flamenco Guitar', -3.0)

vocal.boost(4.5)
guitar.boost(1.5)


## 3. Class Variables & Instance Variables

Örnek değişkenleri (`self.x`) her nesneye özel ayrı kopyalar halinde tutulurken, sınıf değişkenleri doğrudan sınıf gövdesinde tanımlanır ve o sınıftan üretilen tüm nesneler arasında paylaşılır.

**Syntax**

```python
class ClassName:
    shared_var: type = value

    def __init__(self, unique_var: type) -> None:
        self.unique_var = unique_var
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Sınıf değişkenleri sabitler, ortak yapılandırma ayarları veya tüm nesnelerin paylaştığı sayaçlar için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `list` veya `dict` gibi değiştirilebilir (mutable) nesneler sınıf değişkeni olarak tanımlanmamalıdır; bir nesne üzerinden yapılan ekleme/çıkarma diğer tüm nesnelerin verisini etkiler.


In [ ]:
class RenderNode:
    max_temp_c: int = 85

    def __init__(self, node_id: str) -> None:
        self.node_id = node_id
        self.completed_jobs: list[str] = []

    def finish_job(self, job_name: str) -> None:
        self.completed_jobs.append(job_name)


node_1: RenderNode = RenderNode('gpu-01')
node_2: RenderNode = RenderNode('gpu-02')

node_1.finish_job('scene_01.exr')
print(node_1.completed_jobs, node_1.max_temp_c)
print(node_2.completed_jobs, node_2.max_temp_c)


## 4. Getters & Setters (`@property`)

Sınıf niteliklerine dışarıdan doğrudan erişim yerine kontrollü okuma, doğrulama (validation) veya hesaplanmış değer döndürme mekanizması sağlayan dekoratör yapısıdır.

**Syntax**
```python
class ClassName:
    @property
    def attr(self) -> type:
        return self._attr

    @attr.setter
    def attr(self, value: type) -> None:
        self._attr = value

    @attr.deleter
    def attr(self) -> None:
        del self._attr
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Değer atanırken sınır/tip kontrolü yapılması gerektiğinde veya bir niteliğin salt okunur (read-only) olması istendiğinde kullanılır.
* **Dikkat Edilmesi Gerekenler:** Setter içinde `self.attr = value` yerine gizli değişkene (`self._attr = value`) atama yapılmalıdır; aksi halde sonsuz özyineleme (`RecursionError`) oluşur.


In [ ]:
class BatteryPack:
    def __init__(self, voltage: float) -> None:
        self.voltage = voltage

    @property
    def voltage(self) -> float:
        return self._voltage

    @voltage.setter
    def voltage(self, value: float) -> None:
        if not 0.0 <= value <= 4.2:
            raise ValueError('Li-Ion cell voltage must be between 0.0V and 4.2V')
        self._voltage = value

    @property
    def is_charged(self) -> bool:
        return self._voltage >= 4.1


cell: BatteryPack = BatteryPack(3.7)
cell.voltage = 4.15
print(cell.voltage, cell.is_charged)


## 5. `__init__()` & Constructors

`__init__()`, bir sınıftan yeni nesne türetildiğinde otomatik olarak tetiklenen ve nesnenin başlangıç niteliklerini yapılandıran başlatıcı (initializer) metottur.

**Syntax**
```python
class ClassName:
    def __init__(self, param1: type, param2: type = default) -> None:
        self.param1 = param1
        self.param2 = param2
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Nesne oluşturulurken zorunlu veya varsayılan başlangıç parametrelerini almak ve ilk kurulum mantığını çalıştırmak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `__init__()` hiçbir zaman değer döndürmez (`return` kullanılmaz, dönüş tipi `-> None` olmalıdır).


In [ ]:
class CameraRig:
    def __init__(self, sensor: str, fps: int = 24) -> None:
        self.sensor = sensor
        self.fps = fps
        self.frame_duration_ms = round(1000 / fps, 2)


main_cam: CameraRig = CameraRig('Full Frame', 60)
backup_cam: CameraRig = CameraRig('Super 35')

print(main_cam.sensor, main_cam.frame_duration_ms)
print(backup_cam.sensor, backup_cam.frame_duration_ms)


## 6. `__str__()` & `__repr__()`

Nesnenin metinsel temsillerini özelleştiren dunder metotlarıdır: `__str__()` son kullanıcı için okunabilir çıktı üretirken, `__repr__()` geliştiriciler ve hata ayıklama için kesin teknik temsil üretir.

**Syntax**
```python
class ClassName:
    def __str__(self) -> str:
        return 'Human-readable output'

    def __repr__(self) -> str:
        return "ClassName(attr='value')"
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** `print(obj)` ve `f'{obj}'` ifadeleri `__str__()` metodunu; etkileşimli konsol, koleksiyon içi gösterimler ve `repr(obj)` çağrıları `__repr__()` metodunu tetikler.
* **Dikkat Edilmesi Gerekenler:** Sadece birini tanımlayacaksanız `__repr__()` tanımlayın; `__str__()` tanımlı olmadığında Python otomatik olarak `__repr__()` çıktısını kullanır.


In [ ]:
class NetworkPort:
    def __init__(self, protocol: str, number: int) -> None:
        self.protocol = protocol
        self.number = number

    def __str__(self) -> str:
        return f'{self.protocol.upper()} on port {self.number}'

    def __repr__(self) -> str:
        return f"NetworkPort(protocol='{self.protocol}', number={self.number})"


ssh: NetworkPort = NetworkPort('tcp', 22)
print(str(ssh))
print(repr(ssh))
print([ssh])


## 7. `__eq__()`

İki nesne arasında `==` operatörü kullanıldığında değer eşitliğinin hangi kriterlere göre belirleneceğini tanımlayan dunder metodudur.

**Syntax**
```python
class ClassName:
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, ClassName):
            return NotImplemented
        return self.attr == other.attr
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Farklı bellek adreslerindeki iki nesnenin taşıdıkları verilere göre eşit sayılması gerektiğinde kullanılır.
* **Dikkat Edilmesi Gerekenler:** Farklı bir tipteki nesneyle karşılaştırıldığında `AttributeError` fırlatmamak için `isinstance` kontrolü yapıp `NotImplemented` döndürülmelidir. `__eq__()` tanımlanan sınıflar varsayılan olarak hashlenemez (`unhashable`) hale gelir.


In [ ]:
class FirmwareVersion:
    def __init__(self, major: int, minor: int) -> None:
        self.major = major
        self.minor = minor

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, FirmwareVersion):
            return NotImplemented
        return (self.major, self.minor) == (other.major, other.minor)


v1: FirmwareVersion = FirmwareVersion(2, 4)
v2: FirmwareVersion = FirmwareVersion(2, 4)
v3: FirmwareVersion = FirmwareVersion(3, 0)

print(v1 == v2)
print(v1 == v3)
print(v1 == '2.4')


## 8. Methods vs Functions

Fonksiyon (`function`) bağımsız olarak tanımlanıp doğrudan adıyla çağrılan kod bloğu iken, metot (`method`) bir sınıfın gövdesi içinde tanımlanan ve o sınıfa veya nesneye bağlı olarak çağrılan fonksiyondur.

**Syntax**
```python
def standalone_func(arg: type) -> ReturnType:
    ...

class ClassName:
    def bound_method(self, arg: type) -> ReturnType:
        ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Bir nesnenin iç durumunu (`self`) okuyan veya değiştiren işlemler metot olarak; nesne durumundan bağımsız genel işlemler fonksiyon olarak yazılır.
* **Dikkat Edilmesi Gerekenler:** Metotlar bir nesne veya sınıf referansı olmadan doğrudan isimleriyle çağrılamazlar.


In [ ]:
def db_to_linear(db: float) -> float:
    return round(10 ** (db / 20), 4)


class MixerBus:
    def __init__(self, label: str, db: float) -> None:
        self.label = label
        self.db = db

    def linear_gain(self) -> float:
        return db_to_linear(self.db)


print(db_to_linear(-6.0))
master_bus: MixerBus = MixerBus('Master', -6.0)
print(master_bus.linear_gain())


## 9. Private & Protected Members

Python'da gerçek erişim belirteçleri (`private`/`protected` anahtar kelimeleri) yoktur; bunun yerine tek alt çizgi (`_attr`) ile korumalı (protected), çift alt çizgi (`__attr`) ile gizli (private / name mangling) isimlendirme kuralları kullanılır.

**Syntax**
```python
class ClassName:
    def __init__(self) -> None:
        self.public_attr = 1
        self._protected_attr = 2
        self.__private_attr = 3
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** `_protected` yalnızca sınıf ve alt sınıfların kullanması gereken üyeler için; `__private` ise alt sınıflarda isim çakışmasını önlemek ve dışarıdan doğrudan erişimi zorlaştırmak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Python `__private` üyeleri arka planda `_ClassName__private` biçiminde yeniden adlandırır (name mangling). Dışarıdan bu mangled isimle erişim teknik olarak mümkün olsa da kesinlikle kaçınılmalıdır.


In [ ]:
class RouterConfig:
    def __init__(self, ssid: str, channel: int, admin_pin: str) -> None:
        self.ssid = ssid
        self._channel = channel
        self.__admin_pin = admin_pin

    def __validate_pin(self, pin: str) -> bool:
        return pin == self.__admin_pin

    def unlock(self, pin: str) -> bool:
        return self.__validate_pin(pin)


class MeshRouter(RouterConfig):
    def switch_channel(self, new_channel: int) -> None:
        self._channel = new_channel


router: MeshRouter = MeshRouter('Studio_5G', 36, '9482')
router.switch_channel(44)
print(router.ssid, router._channel, router.unlock('9482'))


## 10. Inheritance

Kalıtım (Inheritance), bir alt sınıfın (subclass) mevcut bir üst sınıfın (base/parent class) tüm nitelik ve metotlarını devralarak genişletmesini veya özelleştirmesini sağlar.

**Syntax**
```python
class BaseClass:
    ...

class SubClass(BaseClass):
    ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Sınıflar arasında net bir "bir türüdür" (is-a) ilişkisi varsa ve ortak davranışların tek merkezde toplanması gerekiyorsa kullanılır.
* **Dikkat Edilmesi Gerekenler:** Derin kalıtım zincirleri kodu karmaşıklaştırır; "sahiptir" (has-a) ilişkilerinde kalıtım yerine bileşim (composition) tercih edilmelidir.


In [ ]:
class StorageDevice:
    def __init__(self, mount_point: str, capacity_gb: int) -> None:
        self.mount_point = mount_point
        self.capacity_gb = capacity_gb

    def mount(self) -> None:
        print(f'Mounted {self.mount_point} ({self.capacity_gb} GB)')


class NvmeDrive(StorageDevice):
    def __init__(self, mount_point: str, capacity_gb: int, pcie_gen: int) -> None:
        super().__init__(mount_point, capacity_gb)
        self.pcie_gen = pcie_gen

    def trim(self) -> None:
        print(f'TRIM executed on Gen{self.pcie_gen} drive at {self.mount_point}')


ssd: NvmeDrive = NvmeDrive('/mnt/nvme0', 1000, 4)
ssd.mount()
ssd.trim()


## 11. `super()`

`super()`, alt sınıf içerisinden üst sınıfın (veya çoklu kalıtımda MRO sırasındaki bir sonraki sınıfın) metotlarına ve `__init__` başlatıcısına erişmek için kullanılan yerleşik fonksiyondur.

**Syntax**
```python
class SubClass(BaseClass):
    def __init__(self, arg: type) -> None:
        super().__init__(arg)

    def method(self) -> None:
        super().method()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Üst sınıfın `__init__` kurulumunu çalıştırmak veya üst sınıfta tanımlı bir metodu tamamen ezmek yerine mevcut davranışının üzerine ekleme yapmak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Python 3'te `super(SubClass, self)` yerine parametresiz `super()` kullanılmalıdır. Çoklu kalıtımda MRO (Method Resolution Order) sırasını doğru takip etmek için üst sınıf adını doğrudan yazmak (`BaseClass.method(self)`) yerine `super()` kullanılmalıdır.


In [ ]:
class Logger:
    def write(self, message: str) -> None:
        print(f'[LOG] {message}')


class TimestampLogger(Logger):
    def write(self, message: str) -> None:
        super().write(f'2026-11-24T21:00:00Z | {message}')


log: TimestampLogger = TimestampLogger()
log.write('Service started')


## 12. `@classmethod` & `@staticmethod`

`@classmethod` ilk parametre olarak sınıfın kendisini (`cls`) alırken, `@staticmethod` ne nesneye (`self`) ne de sınıfa (`cls`) ihtiyaç duyan bağımsız yardımcı metotlar tanımlar.

**Syntax**
```python
class ClassName:
    @classmethod
    def alt_constructor(cls, arg: type) -> 'ClassName':
        return cls(arg)

    @staticmethod
    def helper(arg: type) -> ReturnType:
        ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** `@classmethod` alternatif yapıcı metotlar (factory methods) oluşturmak veya sınıf değişkenlerini değiştirmek için; `@staticmethod` ise sınıfın mantıksal kapsamına ait yardımcı fonksiyonları gruplamak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `@classmethod` içinde örnek değişkenlerine (`self.x`) erişilemez. `@staticmethod` hiçbir sınıf veya örnek verisine erişmiyorsa modül düzeyinde düz bir fonksiyon olarak yazmak çoğu zaman daha sadedir.


In [ ]:
class VideoPreset:
    def __init__(self, width: int, height: int, fps: int) -> None:
        self.width = width
        self.height = height
        self.fps = fps

    @classmethod
    def from_resolution_str(cls, res: str, fps: int = 30) -> 'VideoPreset':
        w_str, h_str = res.lower().split('x')
        return cls(int(w_str), int(h_str), fps)

    @staticmethod
    def is_16_by_9(width: int, height: int) -> bool:
        return width * 9 == height * 16


preset: VideoPreset = VideoPreset.from_resolution_str('1920x1080', 60)
print(preset.width, preset.height, preset.fps)
print(VideoPreset.is_16_by_9(preset.width, preset.height))


> **Python 3.11+ Geliştirmesi:** Kendi sınıf tipini döndüren `@classmethod` imzalarında sınıf adını tırnak içine (`'VideoPreset'`) almak yerine `from typing import Self` içe aktarılarak doğrudan `-> Self:` dönüş tipi kullanılabilir.


In [ ]:
from typing import Self


class ModernPreset:
    def __init__(self, width: int, height: int) -> None:
        self.width = width
        self.height = height

    @classmethod
    def uhd_4k(cls) -> Self:
        return cls(3840, 2160)


uhd: ModernPreset = ModernPreset.uhd_4k()
print(uhd.width, uhd.height)


## 13. `@abstractmethod` (`abc` Modülü)

Soyut temel sınıflarda (`ABC`) tanımlanan ve tüm alt sınıflar tarafından mutlaka uygulanması (override edilmesi) zorunlu tutulan arayüz metotlarıdır.

**Syntax**
```python
from abc import ABC, abstractmethod

class BaseInterface(ABC):
    @abstractmethod
    def required_method(self) -> ReturnType:
        ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Ortak bir taban sınıftan türeyen tüm alt sınıfların belirli metotları eksiksiz implemente etmesini garanti altına almak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Soyut metot içeren bir sınıftan doğrudan nesne türetilemez (`TypeError`). `@abstractmethod` başka bir dekoratörle (`@property`, `@classmethod`, `@staticmethod`) birlikte kullanıldığında her zaman en içteki (en alttaki) dekoratör olmalıdır.


In [ ]:
from abc import ABC, abstractmethod


class AudioEffect(ABC):
    @property
    @abstractmethod
    def effect_name(self) -> str:
        ...

    @abstractmethod
    def process(self, signal: float) -> float:
        ...


class HardClipper(AudioEffect):
    def __init__(self, threshold: float) -> None:
        self.threshold = threshold

    @property
    def effect_name(self) -> str:
        return 'Hard Clipper'

    def process(self, signal: float) -> float:
        return max(-self.threshold, min(self.threshold, signal))


clipper: HardClipper = HardClipper(0.8)
print(clipper.effect_name, clipper.process(1.25))


## 14. Protocols (`typing.Protocol`)

Açık bir kalıtım bağı (`class B(A)`) kurmaya gerek kalmadan, yalnızca belirli nitelik ve metot imzalarını barındıran sınıfların statik tip denetleyiciler tarafından uyumlu kabul edilmesini sağlayan yapısal alt tipleme (structural subtyping / duck typing) mekanizmasıdır.

**Syntax**
```python
from typing import Protocol

class InterfaceProtocol(Protocol):
    attr: type

    def action(self) -> ReturnType:
        ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Birbirinden bağımsız sınıfları veya üçüncü parti kütüphane nesnelerini ortak bir taban sınıftan türetmeden aynı arayüz altında kabul etmek için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `Protocol` varsayılan olarak çalışma zamanında (runtime) zorlayıcı değildir; statik analiz araçları (Pyright, Mypy) tarafından denetlenir. Çalışma zamanında `isinstance()` kontrolü gerekiyorsa sınıf `@runtime_checkable` ile dekore edilmelidir.


In [ ]:
from typing import Protocol


class Serializable(Protocol):
    def to_payload(self) -> dict[str, object]:
        ...


class SensorReading:
    def __init__(self, temp_c: float) -> None:
        self.temp_c = temp_c

    def to_payload(self) -> dict[str, object]:
        return {'temperature': self.temp_c}


class UserEvent:
    def __init__(self, action: str) -> None:
        self.action = action

    def to_payload(self) -> dict[str, object]:
        return {'event': self.action}


def transmit(item: Serializable) -> None:
    print('Sending:', item.to_payload())


transmit(SensorReading(23.4))
transmit(UserEvent('login'))


## 15. `__init__()` vs `__new__()`

`__new__()` bellekte yeni nesne örneğini oluşturan ve döndüren gerçek yapıcı (constructor) statik metot iken, `__init__()` `__new__()` tarafından oluşturulan bu nesneyi alıp başlangıç niteliklerini dolduran başlatıcıdır.

**Syntax**
```python
class ClassName:
    def __new__(cls, *args, **kwargs):
        instance = super().__new__(cls)
        return instance

    def __init__(self, *args, **kwargs) -> None:
        ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Singleton tasarım deseni uygulamak, `str`/`int`/`tuple` gibi değişmez (immutable) yerleşik tiplerden alt sınıf türetmek veya koşullara göre farklı bir sınıf örneği döndürmek için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `__new__()` mutlaka oluşturulan nesneyi `return` etmelidir; `None` veya farklı bir sınıf örneği döndürürse ilgili sınıfın `__init__()` metodu çalıştırılmaz. Standart nesne kurulumlarında `__new__()` yerine her zaman `__init__()` kullanılmalıdır.


In [ ]:
class DatabasePool:
    _instance: 'DatabasePool | None' = None

    def __new__(cls, dsn: str) -> 'DatabasePool':  # noqa: PYI034
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.dsn = dsn
        return cls._instance


pool_1: DatabasePool = DatabasePool('postgresql://localhost:5432/app')
pool_2: DatabasePool = DatabasePool('postgresql://localhost:5432/other')

print(pool_1 is pool_2)
print(pool_2.dsn)
